In [2]:
%pip install requests --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import subprocess, json, os, pathlib, time, requests

###############################################################################
# Configuration
###############################################################################
RESOURCE_GROUP_NAME = os.environ.get("RESOURCE_GROUP_NAME", "rg-vk-cwydcosmos")  # ← Replace or set RESOURCE_GROUP_NAME env var

REPO_ROOT = pathlib.Path.cwd()
SEARCH_DATA_FILE_DEFAULT       = REPO_ROOT / "azure_search_data.json"
SEARCH_DATA_FILE_VECTOR_SEMANTIC = REPO_ROOT / "azure_search_data_UseVectorization_SemanticSearch.json"
SAMPLE_DOC_FILE  = REPO_ROOT / "PerksPlus.pdf"

for label, fp in [
    ("Search data (default)",          SEARCH_DATA_FILE_DEFAULT),
    ("Search data (vector+semantic)",  SEARCH_DATA_FILE_VECTOR_SEMANTIC),
    ("Sample doc",                     SAMPLE_DOC_FILE),
]:
    if not fp.exists():
        raise FileNotFoundError(f"❌ {label} file not found: {fp}")

###############################################################################
# Helper — run Azure CLI commands
###############################################################################
def az(cmd: str):
    """Run an Azure CLI command and return parsed JSON (or raw text)."""
    result = subprocess.run(f"az {cmd}", shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"az {cmd}\n{result.stderr.strip()}")
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        return result.stdout.strip()

###############################################################################
# Step 1 — Verify Azure CLI login
###############################################################################
account = az("account show")
SUBSCRIPTION_ID   = account["id"]
SUBSCRIPTION_NAME = account["name"]

user_type = account["user"]["type"]
user_name = account["user"]["name"]
if user_type == "user":
    USER_DISPLAY_NAME = az("ad signed-in-user show")["displayName"]
else:
    USER_DISPLAY_NAME = az(f'ad sp show --id "{user_name}"')["displayName"]

az(f'group show --name "{RESOURCE_GROUP_NAME}"')

###############################################################################
# Step 2 — Discover Search service & Storage account
###############################################################################
search_services = az(f'search service list --resource-group "{RESOURCE_GROUP_NAME}" -o json')
if not search_services:
    raise RuntimeError(f"❌ No Azure Search service found in '{RESOURCE_GROUP_NAME}'.")
AZURE_SEARCH_SERVICE = search_services[0]["name"]

storage_accounts = az(f'storage account list --resource-group "{RESOURCE_GROUP_NAME}" -o json')
if not storage_accounts:
    raise RuntimeError(f"❌ No Storage Account found in '{RESOURCE_GROUP_NAME}'.")
AZURE_BLOB_ACCOUNT_NAME = storage_accounts[0]["name"]

print(f"✅ Search: {AZURE_SEARCH_SERVICE} | Storage: {AZURE_BLOB_ACCOUNT_NAME}")

###############################################################################
# Step 2b — Discover App Service & choose data file based on app settings
###############################################################################
web_apps = az(f'webapp list --resource-group "{RESOURCE_GROUP_NAME}" -o json')
if not web_apps:
    raise RuntimeError(f"❌ No App Service found in '{RESOURCE_GROUP_NAME}'.")
APP_SERVICE_NAME = web_apps[0]["name"]
print(f"✅ App Service: {APP_SERVICE_NAME}")

app_settings_list = az(
    f'webapp config appsettings list --name "{APP_SERVICE_NAME}" '
    f'--resource-group "{RESOURCE_GROUP_NAME}" -o json'
)
app_settings = {s["name"]: s["value"] for s in app_settings_list}

USE_INTEGRATED_VECTORIZATION = app_settings.get("AZURE_SEARCH_USE_INTEGRATED_VECTORIZATION", "").strip().lower() == "true"
USE_SEMANTIC_SEARCH          = app_settings.get("AZURE_SEARCH_USE_SEMANTIC_SEARCH", "").strip().lower() == "true"

if USE_INTEGRATED_VECTORIZATION and USE_SEMANTIC_SEARCH:
    SEARCH_DATA_FILE = SEARCH_DATA_FILE_VECTOR_SEMANTIC
    print("📄 Both AZURE_SEARCH_USE_INTEGRATED_VECTORIZATION and AZURE_SEARCH_USE_SEMANTIC_SEARCH are true.")
    print(f"   → Using data file: {SEARCH_DATA_FILE.name}")
else:
    SEARCH_DATA_FILE = SEARCH_DATA_FILE_DEFAULT
    print(f"📄 AZURE_SEARCH_USE_INTEGRATED_VECTORIZATION={app_settings.get('AZURE_SEARCH_USE_INTEGRATED_VECTORIZATION', '<not set>')}")
    print(f"   AZURE_SEARCH_USE_SEMANTIC_SEARCH={app_settings.get('AZURE_SEARCH_USE_SEMANTIC_SEARCH', '<not set>')}")
    print(f"   → Using data file: {SEARCH_DATA_FILE.name}")

###############################################################################
# Step 3 — Enable public access (wrapped in try/finally for cleanup)
###############################################################################
SEARCH_ENABLED = False
STORAGE_ENABLED = False

try:
    # Check search service
    search_public = search_services[0].get("publicNetworkAccess", "enabled").lower()
    if search_public == "disabled":
        print("🔓 Search service public access is disabled — enabling...")

        # Get current identity configuration
        identity_type = search_services[0].get("identity", {}).get("type", "None")
        print(f"Current identity type: {identity_type}")

        # For UserAssigned identities, use REST API to preserve configuration
        if "UserAssigned" in identity_type:
            print("Using REST API to preserve UserAssigned identity configuration...")

            # Get the complete current configuration
            current_config = search_services[0]
            identity_block = current_config.get("identity", {})

            # Build PATCH request body
            patch_body = {
                "properties": {
                    "publicNetworkAccess": "enabled"
                },
                "identity": identity_block
            }

            # Use az rest to update via Management API
            api_url = (f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}/"
                      f"resourceGroups/{RESOURCE_GROUP_NAME}/providers/Microsoft.Search/"
                      f"searchServices/{AZURE_SEARCH_SERVICE}?api-version=2024-03-01-preview")

            with open("patch_body.json", "w") as f:
                json.dump(patch_body, f)

            az(f'rest --method PATCH --uri "{api_url}" --body @patch_body.json')
            pathlib.Path("patch_body.json").unlink()
            print("✅ Public access enabled for Azure Search service")
        else:
            # For other identity types, use standard CLI
            az(f'search service update --name "{AZURE_SEARCH_SERVICE}" '
               f'--resource-group "{RESOURCE_GROUP_NAME}" --public-access enabled')
            print("✅ Public access enabled for Azure Search service")

        SEARCH_ENABLED = True

    # Check storage account
    storage_public = storage_accounts[0].get("publicNetworkAccess", "Enabled")
    storage_default_action = storage_accounts[0].get("networkRuleSet", {}).get("defaultAction", "Allow")
    if storage_public == "Disabled" or storage_default_action == "Deny":
        print("🔓 Storage account public access is disabled — enabling...")
        az(f'storage account update --name "{AZURE_BLOB_ACCOUNT_NAME}" '
           f'--resource-group "{RESOURCE_GROUP_NAME}" --public-network-access Enabled --output none')
        az(f'storage account update --name "{AZURE_BLOB_ACCOUNT_NAME}" '
           f'--resource-group "{RESOURCE_GROUP_NAME}" --default-action Allow --output none')
        STORAGE_ENABLED = True
        print("✅ Storage account public access enabled.")

    if SEARCH_ENABLED or STORAGE_ENABLED:
        print("⏳ Waiting 7 minutes for network changes to propagate...")
        time.sleep(420)  # Initial 7 minute wait

        # Verify Search service public access is enabled
        if SEARCH_ENABLED:
            max_additional_wait = 600  # Additional 10 minutes (total 17 min)
            check_interval = 60  # Check every 60 seconds
            elapsed = 0

            print("🔍 Verifying Search service public access...")
            while elapsed < max_additional_wait:
                search_status = az(f'search service show --name "{AZURE_SEARCH_SERVICE}" '
                                  f'--resource-group "{RESOURCE_GROUP_NAME}"')
                current_access = search_status.get("publicNetworkAccess", "").lower()

                if current_access == "enabled":
                    print(f"✅ Search service public access confirmed enabled (total wait: {420 + elapsed} seconds)")
                    break

                print(f"   Still verifying... (checking again in 60s)")
                time.sleep(check_interval)
                elapsed += check_interval
            else:
                print(f"⚠️ Could not confirm Search service access after {300 + max_additional_wait}s total, proceeding anyway...")


    ###########################################################################
    # Step 4 — Upload sample doc to Blob Storage (triggers index creation)
    ###########################################################################
    STORAGE_KEY = az(
        f'storage account keys list --account-name "{AZURE_BLOB_ACCOUNT_NAME}" '
        f'--resource-group "{RESOURCE_GROUP_NAME}" --query "[0].value" -o tsv'
    )
    az(f'storage blob upload --account-name "{AZURE_BLOB_ACCOUNT_NAME}" '
       f'--account-key "{STORAGE_KEY}" --container-name "documents" '
       f'--name "{SAMPLE_DOC_FILE.name}" --file "{SAMPLE_DOC_FILE}" --overwrite')
    print("✅ Document uploaded to blob storage.")

    ###########################################################################
    # Step 5 — Get Search admin key & wait for index auto-creation
    ###########################################################################
    AZURE_SEARCH_KEY = az(
        f'search admin-key show --service-name "{AZURE_SEARCH_SERVICE}" '
        f'--resource-group "{RESOURCE_GROUP_NAME}" --query primaryKey -o tsv'
    )

    API_VERSION = "2024-07-01"
    HEADERS = {"Content-Type": "application/json", "api-key": AZURE_SEARCH_KEY}

    # Test connectivity first with retry logic
    print("🔍 Testing connectivity to Search service endpoint...")
    connectivity_established = False
    max_connectivity_attempts = 20  # Try for up to 10 minutes

    for conn_attempt in range(1, max_connectivity_attempts + 1):
        try:
            resp = requests.get(
                f"https://{AZURE_SEARCH_SERVICE}.search.windows.net/indexes?api-version={API_VERSION}&$select=name",
                headers=HEADERS,
                timeout=30
            )
            if resp.status_code in (200, 404):  # 404 is OK - service is reachable, just no indexes yet
                print(f"✅ Search service endpoint is accessible (attempt {conn_attempt})")
                connectivity_established = True
                break
        except (requests.exceptions.ConnectTimeout, requests.exceptions.ConnectionError) as e:
            if conn_attempt < max_connectivity_attempts:
                print(f"   Connection attempt {conn_attempt} failed, retrying in 30s...")
                time.sleep(30)
            else:
                raise RuntimeError(f"❌ Cannot connect to Search service endpoint after {max_connectivity_attempts} attempts. "
                                 f"Network changes may need more time to propagate.")

    if not connectivity_established:
        raise RuntimeError("❌ Search service endpoint is not accessible")

    AZURE_SEARCH_INDEX = None
    INDEX_PREFIX = "index"
    print(f"🔍 Waiting for index with prefix '{INDEX_PREFIX}' to be auto-created (up to 15 min)...")
    for attempt in range(1, 31):
        try:
            resp = requests.get(
                f"https://{AZURE_SEARCH_SERVICE}.search.windows.net/indexes?api-version={API_VERSION}&$select=name",
                headers=HEADERS,
                timeout=30
            )
            if resp.status_code == 200:
                indexes = resp.json().get("value", [])
                matching = [idx for idx in indexes if idx["name"].startswith(INDEX_PREFIX)]
                if matching:
                    AZURE_SEARCH_INDEX = matching[0]["name"]
                    print(f"✅ Found index: '{AZURE_SEARCH_INDEX}'")
                    break
                elif indexes:
                    names = [idx["name"] for idx in indexes]
                    print(f"   Found indexes {names}, but none start with '{INDEX_PREFIX}'. Retrying...")
            if attempt == 30:
                raise RuntimeError(f"❌ No index with prefix '{INDEX_PREFIX}' found after 15 minutes.")
        except (requests.exceptions.ConnectTimeout, requests.exceptions.ConnectionError) as e:
            print(f"   Connection issue on attempt {attempt}, will retry...")

        time.sleep(30)

    ###########################################################################
    # Step 6 — Populate Azure Search index with sample data
    ###########################################################################
    with open(SEARCH_DATA_FILE, "r", encoding="utf-8") as f:
        documents = json.load(f)

    schema_resp = requests.get(
        f"https://{AZURE_SEARCH_SERVICE}.search.windows.net/indexes/{AZURE_SEARCH_INDEX}?api-version={API_VERSION}",
        headers=HEADERS
    )
    if schema_resp.status_code != 200:
        raise RuntimeError(f"❌ Failed to get index schema: {schema_resp.status_code}, {schema_resp.text}")
    index_fields = {f["name"] for f in schema_resp.json()["fields"]}
    cleaned_docs = [{k: v for k, v in doc.items() if k in index_fields} for doc in documents]

    BATCH_SIZE = 1000
    UPLOAD_ENDPOINT = (
        f"https://{AZURE_SEARCH_SERVICE}.search.windows.net"
        f"/indexes/{AZURE_SEARCH_INDEX}/docs/index?api-version={API_VERSION}"
    )
    total_success = 0
    for i in range(0, len(cleaned_docs), BATCH_SIZE):
        batch = cleaned_docs[i:i + BATCH_SIZE]
        payload = {"value": [{"@search.action": "upload", **doc} for doc in batch]}
        response = requests.post(UPLOAD_ENDPOINT, headers=HEADERS, json=payload)
        if response.status_code in (200, 207):
            result = response.json()
            ok = sum(1 for item in result.get("value", []) if item.get("status", False))
            total_success += ok
        else:
            print(f"❌ Batch {i // BATCH_SIZE + 1} failed: {response.status_code}, {response.text}")
            break
    print(f"✅ Import completed! {total_success}/{len(documents)} documents uploaded.")

finally:
    ###########################################################################
    # Step 7 — Restore original access (always runs)
    ###########################################################################
    if SEARCH_ENABLED:
        try:
            print("🔒 Restoring Search service public access to disabled...")

            # Get current configuration with identity
            search_svc = az(f'search service show --name "{AZURE_SEARCH_SERVICE}" '
                           f'--resource-group "{RESOURCE_GROUP_NAME}"')
            identity_type = search_svc.get("identity", {}).get("type", "None")

            # For UserAssigned identities, use REST API
            if "UserAssigned" in identity_type:
                print("Using REST API to preserve UserAssigned identity configuration...")

                identity_block = search_svc.get("identity", {})

                # Build PATCH request body
                patch_body = {
                    "properties": {
                        "publicNetworkAccess": "disabled"
                    },
                    "identity": identity_block
                }

                # Use az rest to update via Management API
                api_url = (f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}/"
                          f"resourceGroups/{RESOURCE_GROUP_NAME}/providers/Microsoft.Search/"
                          f"searchServices/{AZURE_SEARCH_SERVICE}?api-version=2024-03-01-preview")

                with open("patch_body.json", "w") as f:
                    json.dump(patch_body, f)

                az(f'rest --method PATCH --uri "{api_url}" --body @patch_body.json')
                pathlib.Path("patch_body.json").unlink()
            else:
                # For other identity types, use standard CLI
                az(f'search service update --name "{AZURE_SEARCH_SERVICE}" '
                   f'--resource-group "{RESOURCE_GROUP_NAME}" --public-access disabled')

            print("✅ Search service public access restored to disabled.")
        except Exception as e:
            print(f"❌ Search cleanup error: {e}")

    if STORAGE_ENABLED:
        try:
            az(f'storage account update --name "{AZURE_BLOB_ACCOUNT_NAME}" '
               f'--resource-group "{RESOURCE_GROUP_NAME}" --public-network-access Disabled --output none')
            az(f'storage account update --name "{AZURE_BLOB_ACCOUNT_NAME}" '
               f'--resource-group "{RESOURCE_GROUP_NAME}" --default-action Deny --output none')
            print("✅ Storage account public access restored to disabled.")
        except Exception as e:
            print(f"❌ Storage cleanup error: {e}")

    print("\n" + "═" * 63)
    print("  ✅ Import Sample Data (CosmosDB) — Complete")
    print("═" * 63)
    print(f"  Resource Group        : {RESOURCE_GROUP_NAME}")
    print(f"  Azure Search Service  : {AZURE_SEARCH_SERVICE}")
    print(f"  Azure Search Index    : {AZURE_SEARCH_INDEX}")
    print(f"  Storage Account       : {AZURE_BLOB_ACCOUNT_NAME}")
    print(f"  Data File Used        : {SEARCH_DATA_FILE.name}")
    print(f"  User                  : {USER_DISPLAY_NAME} ({user_name})")

✅ Search: srch-vkcwydcosmosyhsof | Storage: stvkcwydcosmosyhsof
✅ App Service: app-vkcwydcosmosyhsof
📄 AZURE_SEARCH_USE_INTEGRATED_VECTORIZATION=false
   AZURE_SEARCH_USE_SEMANTIC_SEARCH=false
   → Using data file: azure_search_data.json
✅ Document uploaded to blob storage.
🔍 Testing connectivity to Search service endpoint...
✅ Search service endpoint is accessible (attempt 1)
🔍 Waiting for index with prefix 'index' to be auto-created (up to 15 min)...
✅ Found index: 'index-vkcwydcosmosyhsof'
✅ Import completed! 1000/1000 documents uploaded.

═══════════════════════════════════════════════════════════════
  ✅ Import Sample Data (CosmosDB) — Complete
═══════════════════════════════════════════════════════════════
  Resource Group        : rg-vk-cwydcosmos
  Azure Search Service  : srch-vkcwydcosmosyhsof
  Azure Search Index    : index-vkcwydcosmosyhsof
  Storage Account       : stvkcwydcosmosyhsof
  Data File Used        : azure_search_data.json
  User                  : Vamshi krishna 